# Stage A & B: Identification & Netlist Generator ⚡

**TL;DR:** In this notebook, we build a real Multi-Agent System using **LangGraph** to parse specs into components (Stage A) and autonomously write a SPICE netlist, run `ngspice`, parse the output, and iteratively adjust the transistor widths to meet a target Gain (Stage B) mimicking AnalogCoder principles.

---

## 🛑 Prerequisites
You need `ngspice` installed on your system to run the simulation node.
```bash
# Debian/Ubuntu
sudo apt-get install ngspice
```
You also need `langgraph`, `langchain`, and an OpenAI/Anthropic API key.

In [ ]:
# !pip install langgraph langchain langchain-openai

import os
import subprocess
import re
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage, SystemMessage

# Set your API Key here
# os.environ["OPENAI_API_KEY"] = "sk-..."

## 1. Stage A: The Identification Agent (Pydantic NER)
Before generating a netlist, we must parse human intent into a structured Component Inventory JSON. We use LangChain's structured output features for this.

In [ ]:
from pydantic import BaseModel, Field

class ComponentInventory(BaseModel):
    circuit_type: str = Field(description="The main type of circuit, e.g., 'Common Source Amplifier' or 'Current Mirror'")
    transistors: List[str] = Field(description="List of transistors required, e.g., ['M1 NMOS', 'M2 PMOS']")
    target_gain: float = Field(description="The target gain in dB if specified.")

llm = ChatOpenAI(model="gpt-4o", temperature=0)
identifier_agent = llm.with_structured_output(ComponentInventory)

# Example invocation:
# spec = "I need a common source amplifier with a target gain of 20dB using a single NMOS."
# inventory = identifier_agent.invoke(spec)
# print(inventory.json(indent=2))


## 2. Define the Shared State
This memory dictionary passes between our Stage B agents. It holds the specs from Stage A, the current code, and the logs.

In [ ]:
class AnalogDesignState(TypedDict):
    inventory: dict       # Populated by Stage A
    target_gain: float
    current_netlist: str
    simulation_log: str
    measured_gain: float
    iterations: int
    status: str

## 3. The Tools (Ngspice Simulator for SKY130)
Agents are useless without tools. This function takes a netlist string, saves it to a file, runs `ngspice`, and extracts the Gain. Note: In a production AnalogCoder flow, this netlist strictly adheres to the SKY130 PDK models.

In [ ]:
def run_ngspice(netlist: str) -> float:
    """Runs ngspice and returns the measured gain."""
    filename = "temp_circuit.sp"
    with open(filename, "w") as f:
        f.write(netlist)
    
    try:
        # Run ngspice in batch mode (-b)
        result = subprocess.run(["ngspice", "-b", filename], capture_output=True, text=True, timeout=10)
        log = result.stdout
        
        # VERY simple regex to find a printed measurement (e.g., "gain = 25.4")
        # In reality, you'd parse raw output or a .print file
        match = re.search(r'gain\s*=\s*([0-9\.]+)', log, re.IGNORECASE)
        if match:
            return float(match.group(1))
        return 0.0 # Failed to parse
    except Exception as e:
        print(f"Simulation Error: {e}")
        return 0.0

## 3. The Agents (Nodes)
Here we define our LangGraph nodes: The **Coder** and the **Simulator**.

In [ ]:
llm = ChatOpenAI(model="gpt-4o", temperature=0.1)

def coder_agent(state: AnalogDesignState):
    """Generates or modifies the SPICE netlist based on the target and previous logs."""
    target = state["target_gain"]
    history = state.get("simulation_log", "No history yet.")
    measured = state.get("measured_gain", 0.0)
    
    system_prompt = f"""
    You are an expert Analog IC Designer. Your goal is to write a SPICE netlist for a Common Source Amplifier.
    The target voltage gain is {target}.
    Last measured gain was: {measured}.
    Previous Simulation Log/Feedback: {history}
    
    Adjust the width (W) of the MOSFET or the load resistor to meet the target gain.
    Output ONLY valid SPICE netlist code. Do not use markdown blocks. Include a .control block that prints 'gain = <value>'.
    """
    
    response = llm.invoke([SystemMessage(content=system_prompt)])
    
    # Clean up output
    netlist = response.content.strip().replace("```spice", "").replace("```", "")
    
    return {
        "current_netlist": netlist,
        "iterations": state.get("iterations", 0) + 1,
        "status": "Netlist generated."
    }

def simulator_node(state: AnalogDesignState):
    """Runs the netlist through the tool."""
    netlist = state["current_netlist"]
    print(f"\n--- Iteration {state['iterations']} ---")
    print("Running Simulation...")
    
    gain = run_ngspice(netlist)
    print(f"Measured Gain: {gain} (Target: {state['target_gain']})")
    
    return {
        "measured_gain": gain,
        "simulation_log": f"Simulated successfully. Gain was {gain}.",
        "status": "Simulation complete."
    }


## 4. The Critic (Conditional Edges)
How does the graph know when to stop? We use a conditional function.

In [ ]:
def check_specs(state: AnalogDesignState):
    """Decides whether to loop back to the Coder or Finish."""
    if state["measured_gain"] >= state["target_gain"]:
        print("\n✅ TARGET MET! Finishing graph.")
        return "end"
    elif state["iterations"] >= 5:
        print("\n❌ MAX ITERATIONS REACHED. Failing.")
        return "end"
    else:
        print("\n🔄 TARGET NOT MET. Looping back to Coder.")
        return "continue"

## 5. Build and Run the Graph

In [ ]:
# Build the Graph
workflow = StateGraph(AnalogDesignState)

workflow.add_node("coder", coder_agent)
workflow.add_node("simulator", simulator_node)

workflow.set_entry_point("coder")
workflow.add_edge("coder", "simulator")

# Add conditional loop
workflow.add_conditional_edges(
    "simulator",
    check_specs,
    {
        "continue": "coder",
        "end": END
    }
)

app = workflow.compile()

# -- Uncomment below to run if you have ngspice and an API key --

# initial_state = {
#     "target_gain": 20.0,
#     "iterations": 0
# }
# final_state = app.invoke(initial_state)
# print("\nFINAL NETLIST:\n")
# print(final_state["current_netlist"])
